# Collecting YouTube Search Results with Selenium

A teaching demo that shows how to scrape YouTube search results from a Google Colab notebook.

- **Author:** Bo Zhao (zhaobo@uw.edu) — Department of Geography, University of Washington
- **Project site:** https://hgis.uw.edu
- **Originally created:** April 14, 2021 · **Last revised:** April 23, 2026

## What this notebook does

1. Installs **Google Chrome** on the Colab VM.
2. Uses **Selenium** to open a YouTube search results page inside a headless browser.
3. Scrolls the page and parses each video card with **BeautifulSoup**.
4. Deduplicates, normalises the fields, and saves the results as a CSV on Google Drive.

## Why Selenium (and not `requests`)?

YouTube's search results page is rendered by JavaScript **after** the HTML arrives, so `requests.get(url).text` returns an almost empty shell. We need a real browser to execute the scripts and reveal the video cards — that is exactly what Selenium drives.

## How to run this notebook

Run the cells **top to bottom**. If you change `QUERY` or `MAX_ITEMS` in Step 2 after the first run, you only need to re-run the last cell of Step 4 — the install, imports, and function definitions above it stay valid for the whole Colab session.

> **Heads-up.** Scraping rendered HTML is fragile: YouTube changes its class names every few months, and `parse_card` in Step 4 may need small updates when that happens. For serious research work, the **YouTube Data API v3** is a better option — see the final cell for a sketch.

## Step 1 — Install Google Chrome

Selenium needs a browser binary. We install **Google Chrome Stable** directly from Google's official `.deb`.

Two packages the old tutorial used do **not** work on current Colab and we no longer touch them:

- `apt-get install chromium` → returns "Package chromium is not available" on Colab's Ubuntu 22.04, because the distro redirects chromium to snap and snap is not functional inside Colab's container.
- Manually downloading `chromedriver_linux64.zip` from `chromedriver.storage.googleapis.com` → pins an ancient version that will not match the Chrome you just installed, which is exactly how you get `DevToolsActivePort file doesn't exist` at startup.

Instead:

- We install `google-chrome-stable` from Google's repo — always the latest stable Chrome.
- We pin **Selenium ≥ 4.15**, whose **Selenium Manager** auto-downloads a matching `chromedriver` the first time `webdriver.Chrome(...)` runs. No manual version juggling.
- `beautifulsoup4` parses the rendered HTML and `pandas` writes the CSV.

In [ ]:
# 1. Install Google Chrome from Google's official .deb.
!apt-get -qq update
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!rm -f google-chrome-stable_current_amd64.deb

# 2. Install the Python libraries. Selenium >= 4.15 ships a reliable Selenium Manager,
#    which auto-downloads a matching chromedriver the first time webdriver.Chrome() runs.
!pip install -q "selenium>=4.15" beautifulsoup4 pandas

# 3. Sanity check — if this prints a version like "Google Chrome 147.0.x", you are good.
!google-chrome --version

## Step 2 — Configure the task

Keep all the knobs that you might want to change at the top of the notebook. The scraping code below reads from these variables, so you can rerun against a new search term without touching the logic.

| Variable | Meaning |
|---|---|
| `QUERY` | the search keywords |
| `MAX_ITEMS` | stop after collecting this many unique videos |
| `MAX_SCROLLS` | safety cap — give up after this many scrolls even if we did not reach `MAX_ITEMS` |
| `SCROLL_PAUSE` | seconds to wait after each scroll for new cards to load |
| `HEADLESS` | must stay `True` on Colab (there is no display); set `False` only if you run this notebook locally |
| `OUTPUT_PATH` | where to save the CSV on Google Drive |

In [ ]:
QUERY        = "standing rock"
MAX_ITEMS    = 100
MAX_SCROLLS  = 20
SCROLL_PAUSE = 2.0
HEADLESS     = True
OUTPUT_PATH  = "/gdrive/My Drive/videos.csv"

## Step 3 — Imports and helpers

Before we scrape, we define three small helpers. Factoring them out keeps the main loop readable and makes each piece easy to test or swap.

**`parse_view_count("1.2K views")` → `1200`**
YouTube shows counts like `"3.4M views"` or `"1,203 views"`. We convert them to integers so pandas can sort and aggregate.

**`parse_relative_time("3 years ago", anchor)` → `datetime`**
YouTube only shows *relative* ages (`"2 weeks ago"`). We anchor them to the moment we scraped, so the CSV carries absolute UTC timestamps — far more useful downstream.

**`build_driver(headless)`** centralises the Chrome options we always want on Colab:

- `binary_location="/usr/bin/google-chrome"` — point Selenium at the Chrome we installed in Step 1.
- `--headless=new` — the modern headless mode (Chrome 109+); the old `--headless` crashes on some pages.
- `--no-sandbox` — required because Colab runs as root.
- `--disable-dev-shm-usage` — **this is the fix for `DevToolsActivePort file doesn't exist`**. Colab's `/dev/shm` is tiny, and Chrome crashes when it tries to use it for renderer IPC.
- `--disable-gpu` — no GPU in the container, saves a warning.
- `--window-size=1280,1800` — give the page a real viewport so lazy-loaded cards render.
- `--lang=en-US` — keep the DOM in English so our selectors and `"N views"` parsing stay consistent.

In [ ]:
import logging
import re
import time
import urllib.parse
from datetime import datetime, timedelta, timezone

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("youtube-scraper")


_VIEW_SUFFIX = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}

def parse_view_count(text):
    """Turn '1.2K views' / '3,401 views' / 'No views' into an int (None if unparseable)."""
    if not text:
        return None
    s = text.lower().replace("views", "").replace("view", "").replace(",", "").strip()
    if s in ("", "no"):
        return 0
    m = re.match(r"^([0-9]*\.?[0-9]+)\s*([kmb]?)$", s)
    if not m:
        return None
    return int(float(m.group(1)) * _VIEW_SUFFIX.get(m.group(2).upper(), 1))


_UNIT_TO_DAYS = {
    "second": 1 / 86400, "minute": 1 / 1440, "hour": 1 / 24,
    "day": 1, "week": 7, "month": 30, "year": 365,
}

def parse_relative_time(text, anchor):
    """'Streamed 3 years ago' + anchor datetime → absolute datetime (None if unparseable)."""
    if not text:
        return None
    cleaned = text.lower().replace("streamed", "").replace("premiered", "").strip()
    m = re.match(r"^(\d+)\s+(second|minute|hour|day|week|month|year)s?\s+ago$", cleaned)
    if not m:
        return None
    qty, unit = int(m.group(1)), m.group(2)
    return anchor - timedelta(days=qty * _UNIT_TO_DAYS[unit])


def build_driver(headless=True):
    opts = Options()
    opts.binary_location = "/usr/bin/google-chrome"
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1280,1800")
    opts.add_argument("--lang=en-US")
    return webdriver.Chrome(options=opts)

## Step 4 — Scrape the search page

Two code cells below:

1. **Definitions** — `parse_card` (extract fields from one `<ytd-video-renderer>` node) and `scrape_youtube_search` (the main loop).
2. **Run** — invoke the scraper and show the first rows. Re-run *only this cell* after editing `QUERY` / `MAX_ITEMS` in Step 2.

Five things in the loop worth noting:

1. **Wait for the first card, not a fixed sleep.** `WebDriverWait(...).until(EC.presence_of_element_located(...))` returns the moment at least one `ytd-video-renderer` appears. Blind `time.sleep(5)` is either too slow or too fast.
2. **Scroll until the page stops growing.** We compare `document.documentElement.scrollHeight` before and after each scroll; when it no longer changes, YouTube has streamed all its results and we stop early. The old code's fixed `range(5)` did not adapt.
3. **Deduplicate with a `set`.** `if video_url in seen:` on a `set` is O(1); the old `list` version was silently quadratic.
4. **Catch narrow exceptions per card.** If one card has unexpected markup, we log and skip — the other cards still land in the CSV. The old `except: pass` also swallowed `KeyboardInterrupt`, which made debugging painful.
5. **Always `driver.quit()` in `finally`.** Releases the Chrome process even if the loop raises. `driver.close()` only closes one tab and leaks the session.

If YouTube changes its markup, `parse_card` is the **only** function you have to update.

In [ ]:
def parse_card(card):
    """Extract one video's fields from a <ytd-video-renderer> soup node, or None on failure."""
    title_tag = card.find("a", id="video-title") \
                or card.find("a", class_="yt-simple-endpoint style-scope ytd-video-renderer")
    if not title_tag or not title_tag.get("href"):
        return None

    channel_tag = card.find("a", class_=re.compile(r"yt-simple-endpoint.*yt-formatted-string"))
    meta_tags = (card.find_all("span", class_="inline-metadata-item style-scope ytd-video-meta-block")
                 or card.find_all("span", class_="style-scope ytd-video-meta-block"))
    desc_tag = card.find("yt-formatted-string",
                         class_="metadata-snippet-text style-scope ytd-video-renderer")

    collected_at = datetime.now(timezone.utc)
    view_raw    = meta_tags[0].get_text(strip=True) if len(meta_tags) > 0 else None
    created_raw = meta_tags[1].get_text(strip=True) if len(meta_tags) > 1 else None

    return {
        "video_url":      "https://www.youtube.com" + title_tag["href"],
        "title":          title_tag.get("title") or title_tag.get_text(strip=True),
        "user_url":       ("https://www.youtube.com" + channel_tag["href"]) if channel_tag and channel_tag.get("href") else None,
        "username":       channel_tag.get_text(strip=True) if channel_tag else None,
        "view_num":       parse_view_count(view_raw),
        "view_num_raw":   view_raw,
        "created_at":     parse_relative_time(created_raw, collected_at),
        "created_at_raw": created_raw,
        "shortdesc":      desc_tag.get_text(" ", strip=True) if desc_tag else None,
        "collected_at":   collected_at,
    }


def scrape_youtube_search(query, max_items=MAX_ITEMS, max_scrolls=MAX_SCROLLS,
                          scroll_pause=SCROLL_PAUSE, headless=HEADLESS):
    url = "https://www.youtube.com/results?search_query=" + urllib.parse.quote_plus(query)
    log.info("opening %s", url)

    driver = build_driver(headless=headless)
    results, seen = [], set()
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer"))
        )

        last_height = 0
        for scroll in range(max_scrolls):
            soup = BeautifulSoup(driver.page_source, "html.parser")
            for card in soup.find_all("ytd-video-renderer"):
                try:
                    row = parse_card(card)
                except Exception as e:
                    log.warning("parse failed: %s", e)
                    continue
                if not row or row["video_url"] in seen:
                    continue
                seen.add(row["video_url"])
                results.append(row)
                if len(results) >= max_items:
                    log.info("reached MAX_ITEMS=%d after %d scroll(s)", max_items, scroll)
                    return pd.DataFrame(results)

            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
            time.sleep(scroll_pause)

            new_height = driver.execute_script("return document.documentElement.scrollHeight")
            log.info("scroll %d/%d · collected %d videos · page height %d",
                     scroll + 1, max_scrolls, len(results), new_height)
            if new_height == last_height:
                log.info("page height stopped growing; no more results to load")
                break
            last_height = new_height

        return pd.DataFrame(results)
    finally:
        driver.quit()

In [ ]:
# Run the scraper. Tweak QUERY / MAX_ITEMS in Step 2 and re-run just this cell
# to collect a new dataset — no need to re-run the definitions above.
df = scrape_youtube_search(QUERY)
log.info("collected %d unique videos", len(df))
df.head()

## Step 5 — Save the results to Google Drive

Mount your Drive once per Colab session, then write the CSV. The file appears in **My Drive** and you can open it from the browser, share it with the class, or load it from another notebook — no extra `files.download(...)` step needed.

In [ ]:
from google.colab import drive
drive.mount("/gdrive")

df.to_csv(OUTPUT_PATH, index=False)
log.info("saved %d rows to %s", len(df), OUTPUT_PATH)

## Option 2 — The YouTube Data API v3

Selenium scraping is great for *learning* how dynamic pages are rendered, but for real research work the official **YouTube Data API v3** is a stronger tool:

- **Stable.** JSON from a documented endpoint, not HTML whose class names change every few months.
- **Richer.** Exact view / like / comment counts, ISO-8601 timestamps, duration, language, category, thumbnails — no string parsing, no `"3 years ago"` approximations.
- **Fast.** Up to 50 results per request, proper pagination via `nextPageToken`, no scrolling or `sleep`.
- **Compliant.** YouTube's Terms of Service generally disallow scraping; API use is permitted under the standard quota (10,000 units per day is plenty for most classroom tasks).

### Sketch of the equivalent workflow

```python
# !pip install -q google-api-python-client
from googleapiclient.discovery import build
from google.colab import userdata   # store your key in Colab Secrets, not in code

yt = build("youtube", "v3", developerKey=userdata.get("YT_API_KEY"))

# 1. search → get video ids (snippet only, cheap)
search = yt.search().list(q="standing rock", type="video",
                          part="snippet", maxResults=50).execute()
ids = [item["id"]["videoId"] for item in search["items"]]

# 2. videos.list → batched call for full stats on those ids
details = yt.videos().list(id=",".join(ids),
                           part="snippet,statistics,contentDetails").execute()
```

### When to pick which

| Situation | Use |
|---|---|
| Learning how dynamic pages, DOM, and browser automation work | **Selenium (this notebook)** |
| Quick one-off dataset for a class demo, small N | Either — Selenium is faster to copy-paste |
| Research paper, reproducible study, N in the thousands | **Data API v3** |
| You need fields the rendered page does not show (exact likes, captions, categories) | **Data API v3** |

For this class we stay with Selenium so you can see the full page-rendering pipeline — but keep the API in mind for your own projects.